In [1]:
# ============================================================
# STAGE 2A — CLINICALTRIALS.GOV RAW INGESTION + INITIAL AUDIT
# ============================================================

from pathlib import Path
import json
import time

import pandas as pd
import requests


# ============================================================
# 1. Configuration
# ============================================================

API_BASE = "https://clinicaltrials.gov/api/v2"
STUDIES_URL = f"{API_BASE}/studies"
VERSION_URL = f"{API_BASE}/version"

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "clinical_trials"
RAW_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_QUERY = "Obesity OR Overweight"

SPONSORS = {
    "novo_nordisk": "Novo Nordisk",
    "eli_lilly": "Eli Lilly and Company",
    "amgen": "Amgen",
    "boehringer_ingelheim": "Boehringer Ingelheim",
}

session = requests.Session()

session.headers.update({
    "User-Agent": "clinical-intelligence-copilot/0.1"
})


# ============================================================
# 2. Save ClinicalTrials.gov data snapshot information
# ============================================================

version_response = session.get(
    VERSION_URL,
    timeout=30
)

version_response.raise_for_status()

version_path = RAW_DIR / "api_version.json"

# Save exact API response bytes
version_path.write_bytes(
    version_response.content
)

version_info = version_response.json()

print("CLINICALTRIALS.GOV DATA SNAPSHOT")
print("=" * 90)
print(json.dumps(version_info, indent=2))


# ============================================================
# 3. Download all matching studies for one sponsor
#
# Each API response page is saved exactly as returned.
# We do NOT filter phase/status/study type yet.
# ============================================================

def download_sponsor_studies(
    sponsor_key,
    sponsor_query,
):

    sponsor_dir = RAW_DIR / sponsor_key
    sponsor_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    all_studies = []
    page_token = None
    page_number = 1

    while True:

        params = {
            "query.cond": CONDITION_QUERY,
            "query.spons": sponsor_query,
            "pageSize": 1000,
            "format": "json",
        }

        if page_token is not None:
            params["pageToken"] = page_token

        response = session.get(
            STUDIES_URL,
            params=params,
            timeout=60
        )

        response.raise_for_status()

        # --------------------------------------------
        # Preserve exact raw API response
        # --------------------------------------------

        raw_path = (
            sponsor_dir
            / f"page_{page_number:03d}.json"
        )

        raw_path.write_bytes(
            response.content
        )

        payload = response.json()

        studies = payload.get(
            "studies",
            []
        )

        all_studies.extend(
            studies
        )

        print(
            f"{sponsor_key:<25} "
            f"page={page_number:<3} "
            f"studies={len(studies):>4}"
        )

        page_token = payload.get(
            "nextPageToken"
        )

        if not page_token:
            break

        page_number += 1

        # Small pause; unnecessary for correctness,
        # but avoids hammering the public API.
        time.sleep(0.2)

    return all_studies


# ============================================================
# 4. Download all four sponsor searches
# ============================================================

studies_by_company = {}

print("\n")
print("DOWNLOADING STUDIES")
print("=" * 90)

for sponsor_key, sponsor_query in SPONSORS.items():

    studies_by_company[sponsor_key] = (
        download_sponsor_studies(
            sponsor_key,
            sponsor_query
        )
    )

    print(
        f"  -> Total returned: "
        f"{len(studies_by_company[sponsor_key]):,}\n"
    )


# ============================================================
# 5. Minimal extraction for AUDIT ONLY
#
# We are deliberately NOT designing the final schema yet.
# ============================================================

def extract_audit_record(
    study,
    searched_company
):

    protocol = study.get(
        "protocolSection",
        {}
    )

    identification = protocol.get(
        "identificationModule",
        {}
    )

    sponsor_module = protocol.get(
        "sponsorCollaboratorsModule",
        {}
    )

    design = protocol.get(
        "designModule",
        {}
    )

    status = protocol.get(
        "statusModule",
        {}
    )

    conditions = protocol.get(
        "conditionsModule",
        {}
    )

    lead_sponsor = sponsor_module.get(
        "leadSponsor",
        {}
    )

    start_struct = status.get(
        "startDateStruct",
        {}
    )

    return {
        "searched_company":
            searched_company,

        "nct_id":
            identification.get("nctId"),

        "brief_title":
            identification.get("briefTitle"),

        "lead_sponsor":
            lead_sponsor.get("name"),

        "lead_sponsor_class":
            lead_sponsor.get("class"),

        "study_type":
            design.get("studyType"),

        "phases":
            design.get("phases", []),

        "overall_status":
            status.get("overallStatus"),

        "start_date":
            start_struct.get("date"),

        "conditions":
            conditions.get(
                "conditions",
                []
            ),
    }


audit_records = []

for company, studies in studies_by_company.items():

    for study in studies:

        audit_records.append(
            extract_audit_record(
                study,
                company
            )
        )

audit = pd.DataFrame(
    audit_records
)


# ============================================================
# 6. Basic integrity checks
# ============================================================

print("\n")
print("INITIAL DATA AUDIT")
print("=" * 90)

print(
    f"Rows returned across searches: "
    f"{len(audit):,}"
)

print(
    f"Unique NCT IDs overall: "
    f"{audit['nct_id'].nunique():,}"
)

duplicate_across_searches = (
    audit["nct_id"].duplicated(
        keep=False
    ).sum()
)

print(
    f"Rows whose NCT ID appears in "
    f"multiple search results: "
    f"{duplicate_across_searches:,}"
)


# ============================================================
# 7. Company-level audit
# ============================================================

for company in SPONSORS:

    subset = audit.loc[
        audit["searched_company"] == company
    ].copy()

    parsed_dates = pd.to_datetime(
        subset["start_date"],
        errors="coerce"
    )

    print("\n")
    print("=" * 90)
    print(company.upper())
    print("=" * 90)

    print(
        f"Studies returned:     "
        f"{len(subset):,}"
    )

    print(
        f"Unique NCT IDs:       "
        f"{subset['nct_id'].nunique():,}"
    )

    print(
        f"Missing NCT IDs:      "
        f"{subset['nct_id'].isna().sum():,}"
    )

    print(
        f"Start-date coverage:  "
        f"{parsed_dates.min()} -> "
        f"{parsed_dates.max()}"
    )


    print("\nLead sponsor names:")
    print(
        subset["lead_sponsor"]
        .value_counts(
            dropna=False
        )
        .head(20)
        .to_string()
    )


    print("\nStudy types:")
    print(
        subset["study_type"]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


    print("\nPhases:")

    phase_counts = (
        subset["phases"]
        .explode()
        .value_counts(
            dropna=False
        )
    )

    print(
        phase_counts.to_string()
    )


    print("\nOverall statuses:")
    print(
        subset["overall_status"]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


# ============================================================
# 8. Missingness relevant to our eventual project
# ============================================================

print("\n")
print("=" * 90)
print("INITIAL FIELD COMPLETENESS")
print("=" * 90)

for col in [
    "nct_id",
    "brief_title",
    "lead_sponsor",
    "study_type",
    "overall_status",
    "start_date",
]:

    missing = (
        audit[col]
        .isna()
        .mean()
        * 100
    )

    print(
        f"{col:<20}: "
        f"{missing:6.2f}% missing"
    )


empty_phase = (
    audit["phases"]
    .apply(
        lambda x:
        not isinstance(x, list)
        or len(x) == 0
    )
    .mean()
    * 100
)

empty_conditions = (
    audit["conditions"]
    .apply(
        lambda x:
        not isinstance(x, list)
        or len(x) == 0
    )
    .mean()
    * 100
)

print(
    f"{'phases':<20}: "
    f"{empty_phase:6.2f}% empty"
)

print(
    f"{'conditions':<20}: "
    f"{empty_conditions:6.2f}% empty"
)


# ============================================================
# 9. Search-overlap audit
#
# Important because query.spons searches sponsor/collaborator
# information, so a study could legitimately match more than
# one company query.
# ============================================================

nct_search_counts = (
    audit.groupby(
        "nct_id",
        observed=True
    )["searched_company"]
    .nunique()
)

overlapping_ncts = (
    nct_search_counts[
        nct_search_counts > 1
    ]
)

print("\n")
print("=" * 90)
print("CROSS-COMPANY SEARCH OVERLAP")
print("=" * 90)

print(
    f"NCT IDs matching >1 company search: "
    f"{len(overlapping_ncts):,}"
)

if len(overlapping_ncts):

    overlap_detail = (
        audit.loc[
            audit["nct_id"].isin(
                overlapping_ncts.index
            ),
            [
                "nct_id",
                "searched_company",
                "lead_sponsor",
                "brief_title",
            ]
        ]
        .sort_values(
            ["nct_id", "searched_company"]
        )
    )

    print(
        overlap_detail
        .head(30)
        .to_string(index=False)
    )


print("\n")
print("=" * 90)
print("STAGE 2A COMPLETE")
print("=" * 90)

print(f"Raw files saved under:")
print(RAW_DIR)

print("\nObjects ready:")
print("  studies_by_company")
print("  audit")

CLINICALTRIALS.GOV DATA SNAPSHOT
{
  "apiVersion": "2.0.5",
  "dataTimestamp": "2026-08-26T09:00:05"
}


DOWNLOADING STUDIES
novo_nordisk              page=1   studies= 270
  -> Total returned: 270

eli_lilly                 page=1   studies= 135
  -> Total returned: 135

amgen                     page=1   studies=  37
  -> Total returned: 37

boehringer_ingelheim      page=1   studies=  38
  -> Total returned: 38



INITIAL DATA AUDIT
Rows returned across searches: 480
Unique NCT IDs overall: 477
Rows whose NCT ID appears in multiple search results: 6


NOVO_NORDISK
Studies returned:     270
Unique NCT IDs:       270
Missing NCT IDs:      0
Start-date coverage:  1997-04-09 00:00:00 -> 2027-02-01 00:00:00

Lead sponsor names:
lead_sponsor
Novo Nordisk A/S                            210
Steno Diabetes Center Copenhagen              3
Inversago Pharma Inc.                         2
University of Pennsylvania                    2
Mayo Clinic                                   2
Hvidovre Un

In [2]:
# ============================================================
# STAGE 2B — CANONICAL TRIAL TABLE + COHORT DEFINITION
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


# ============================================================
# 1. Paths + canonical sponsor mapping
# ============================================================

PROJECT_ROOT = Path("..").resolve()

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_trials"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Exact lead-sponsor names observed in Stage 2A
LEAD_SPONSOR_MAP = {
    "Novo Nordisk A/S": "Novo Nordisk",
    "Eli Lilly and Company": "Eli Lilly",
    "Amgen": "Amgen",
    "Boehringer Ingelheim": "Boehringer Ingelheim",
}


# ============================================================
# 2. Reload raw studies from disk
#
# Important:
# This makes Stage 2B reproducible even after restarting
# the notebook/kernel.
# ============================================================

raw_records = []

for sponsor_dir in RAW_DIR.iterdir():

    if not sponsor_dir.is_dir():
        continue

    searched_company = sponsor_dir.name

    for page_file in sorted(
        sponsor_dir.glob("page_*.json")
    ):

        with open(
            page_file,
            "r",
            encoding="utf-8"
        ) as f:

            payload = json.load(f)

        for study in payload.get(
            "studies",
            []
        ):

            raw_records.append({
                "searched_company":
                    searched_company,

                "study":
                    study,
            })


print("STAGE 2B — CANONICAL COHORT")
print("=" * 100)

print(
    f"Raw search-result rows loaded: "
    f"{len(raw_records):,}"
)


# ============================================================
# 3. Extraction helpers
# ============================================================

def get_nested(
    obj,
    *keys,
    default=None
):
    """Safely retrieve nested dictionary fields."""

    current = obj

    for key in keys:

        if not isinstance(
            current,
            dict
        ):
            return default

        current = current.get(
            key
        )

        if current is None:
            return default

    return current



def extract_interventions(module):

    interventions = module.get(
        "interventions",
        []
    ) if isinstance(module, dict) else []

    names = []
    types = []
    descriptions = []

    for intervention in interventions:

        names.append(
            intervention.get("name")
        )

        types.append(
            intervention.get("type")
        )

        descriptions.append(
            intervention.get(
                "description"
            )
        )

    return (
        [x for x in names if x],
        [x for x in types if x],
        [x for x in descriptions if x],
    )



def extract_outcomes(module, key):

    outcomes = module.get(
        key,
        []
    ) if isinstance(module, dict) else []

    records = []

    for outcome in outcomes:

        records.append({
            "measure":
                outcome.get("measure"),

            "description":
                outcome.get("description"),

            "time_frame":
                outcome.get("timeFrame"),
        })

    return records



def extract_locations(module):

    locations = module.get(
        "locations",
        []
    ) if isinstance(module, dict) else []

    countries = []
    states = []
    cities = []

    for loc in locations:

        if loc.get("country"):
            countries.append(
                loc["country"]
            )

        if loc.get("state"):
            states.append(
                loc["state"]
            )

        if loc.get("city"):
            cities.append(
                loc["city"]
            )

    return {
        "location_count":
            len(locations),

        "countries":
            sorted(set(countries)),

        "states":
            sorted(set(states)),

        "cities":
            sorted(set(cities)),
    }



# ============================================================
# 4. Extract rich trial-level records
# ============================================================

records = []

for raw in raw_records:

    study = raw["study"]

    protocol = study.get(
        "protocolSection",
        {}
    )

    identification = protocol.get(
        "identificationModule",
        {}
    )

    status = protocol.get(
        "statusModule",
        {}
    )

    sponsors = protocol.get(
        "sponsorCollaboratorsModule",
        {}
    )

    description = protocol.get(
        "descriptionModule",
        {}
    )

    conditions_module = protocol.get(
        "conditionsModule",
        {}
    )

    design = protocol.get(
        "designModule",
        {}
    )

    arms = protocol.get(
        "armsInterventionsModule",
        {}
    )

    outcomes = protocol.get(
        "outcomesModule",
        {}
    )

    eligibility = protocol.get(
        "eligibilityModule",
        {}
    )

    contacts_locations = protocol.get(
        "contactsLocationsModule",
        {}
    )


    # --------------------------------------------------------
    # Sponsor
    # --------------------------------------------------------

    lead_sponsor = sponsors.get(
        "leadSponsor",
        {}
    )

    collaborators = sponsors.get(
        "collaborators",
        []
    )

    collaborator_names = [
        c.get("name")
        for c in collaborators
        if c.get("name")
    ]


    # --------------------------------------------------------
    # Interventions
    # --------------------------------------------------------

    (
        intervention_names,
        intervention_types,
        intervention_descriptions,
    ) = extract_interventions(
        arms
    )


    # --------------------------------------------------------
    # Outcomes
    # --------------------------------------------------------

    primary_outcomes = (
        extract_outcomes(
            outcomes,
            "primaryOutcomes"
        )
    )

    secondary_outcomes = (
        extract_outcomes(
            outcomes,
            "secondaryOutcomes"
        )
    )


    # --------------------------------------------------------
    # Locations
    # --------------------------------------------------------

    location_info = (
        extract_locations(
            contacts_locations
        )
    )


    # --------------------------------------------------------
    # Dates
    # --------------------------------------------------------

    start_date = get_nested(
        status,
        "startDateStruct",
        "date"
    )

    primary_completion_date = get_nested(
        status,
        "primaryCompletionDateStruct",
        "date"
    )

    completion_date = get_nested(
        status,
        "completionDateStruct",
        "date"
    )


    # --------------------------------------------------------
    # Enrollment
    # --------------------------------------------------------

    enrollment_info = design.get(
        "enrollmentInfo",
        {}
    )


    # --------------------------------------------------------
    # Study design
    # --------------------------------------------------------

    design_info = design.get(
        "designInfo",
        {}
    )


    record = {

        # Provenance
        "searched_company":
            raw["searched_company"],

        # Identity
        "nct_id":
            identification.get("nctId"),

        "brief_title":
            identification.get("briefTitle"),

        "official_title":
            identification.get("officialTitle"),

        "acronym":
            identification.get("acronym"),


        # Sponsor
        "lead_sponsor":
            lead_sponsor.get("name"),

        "lead_sponsor_class":
            lead_sponsor.get("class"),

        "collaborators":
            collaborator_names,


        # Study classification
        "study_type":
            design.get("studyType"),

        "phases":
            design.get(
                "phases",
                []
            ),

        "overall_status":
            status.get("overallStatus"),

        "status_verified_date":
            status.get(
                "statusVerifiedDate"
            ),


        # Timing
        "start_date":
            start_date,

        "primary_completion_date":
            primary_completion_date,

        "completion_date":
            completion_date,


        # Condition / keywords
        "conditions":
            conditions_module.get(
                "conditions",
                []
            ),

        "keywords":
            conditions_module.get(
                "keywords",
                []
            ),


        # Trial description
        "brief_summary":
            description.get(
                "briefSummary"
            ),

        "detailed_description":
            description.get(
                "detailedDescription"
            ),


        # Intervention
        "intervention_names":
            intervention_names,

        "intervention_types":
            intervention_types,

        "intervention_descriptions":
            intervention_descriptions,


        # Design
        "enrollment":
            enrollment_info.get(
                "count"
            ),

        "enrollment_type":
            enrollment_info.get(
                "type"
            ),

        "allocation":
            design_info.get(
                "allocation"
            ),

        "intervention_model":
            design_info.get(
                "interventionModel"
            ),

        "primary_purpose":
            design_info.get(
                "primaryPurpose"
            ),

        "masking":
            get_nested(
                design_info,
                "maskingInfo",
                "masking"
            ),


        # Outcomes
        "primary_outcomes":
            primary_outcomes,

        "secondary_outcomes":
            secondary_outcomes,


        # Eligibility
        "eligibility_criteria":
            eligibility.get(
                "eligibilityCriteria"
            ),

        "healthy_volunteers":
            eligibility.get(
                "healthyVolunteers"
            ),

        "sex":
            eligibility.get("sex"),

        "minimum_age":
            eligibility.get(
                "minimumAge"
            ),

        "maximum_age":
            eligibility.get(
                "maximumAge"
            ),


        # Geography
        "location_count":
            location_info[
                "location_count"
            ],

        "countries":
            location_info[
                "countries"
            ],

        "states":
            location_info[
                "states"
            ],

        "cities":
            location_info[
                "cities"
            ],
    }

    records.append(record)


trials_raw = pd.DataFrame(
    records
)


# ============================================================
# 5. Resolve cross-search duplicates
#
# Same NCT can appear in multiple company searches.
# Trial content itself is identical; preserve all searches
# separately before collapsing.
# ============================================================

search_matches = (
    trials_raw.groupby(
        "nct_id",
        observed=True
    )["searched_company"]
    .agg(
        lambda x:
        sorted(set(x))
    )
    .rename(
        "matched_searches"
    )
)


trials = (
    trials_raw
    .drop_duplicates(
        subset="nct_id",
        keep="first"
    )
    .drop(
        columns="searched_company"
    )
    .merge(
        search_matches,
        on="nct_id",
        how="left",
        validate="one_to_one"
    )
)


print(
    f"Unique trials after deduplication: "
    f"{len(trials):,}"
)


# ============================================================
# 6. Canonical company ownership
#
# IMPORTANT:
# We use LEAD SPONSOR for pipeline ownership.
#
# A company merely appearing as collaborator/search match
# does not make the trial part of its own pipeline.
# ============================================================

trials[
    "canonical_company"
] = (
    trials["lead_sponsor"]
    .map(
        LEAD_SPONSOR_MAP
    )
)

trials[
    "is_target_company_lead"
] = (
    trials[
        "canonical_company"
    ].notna()
)


# ============================================================
# 7. Cohort flags
#
# Do NOT delete records.
# Add explicit flags so every filtering decision remains
# inspectable and reversible.
# ============================================================

trials[
    "is_interventional"
] = (
    trials["study_type"]
    == "INTERVENTIONAL"
)


def contains_phase(
    phases,
    phase
):

    return (
        isinstance(phases, list)
        and phase in phases
    )


trials[
    "has_phase_1"
] = (
    trials["phases"]
    .apply(
        lambda x:
        contains_phase(
            x,
            "PHASE1"
        )
    )
)

trials[
    "has_phase_2"
] = (
    trials["phases"]
    .apply(
        lambda x:
        contains_phase(
            x,
            "PHASE2"
        )
    )
)

trials[
    "has_phase_3"
] = (
    trials["phases"]
    .apply(
        lambda x:
        contains_phase(
            x,
            "PHASE3"
        )
    )
)

trials[
    "has_phase_4"
] = (
    trials["phases"]
    .apply(
        lambda x:
        contains_phase(
            x,
            "PHASE4"
        )
    )
)


# Core competitive-development cohort:
#
# target-company lead sponsor
# + interventional
# + Phase 2 or Phase 3
#
# We deliberately DO NOT remove completed/terminated/withdrawn
# studies yet because historical pipeline evolution is useful.

trials[
    "is_core_pipeline"
] = (
    trials[
        "is_target_company_lead"
    ]
    &
    trials[
        "is_interventional"
    ]
    &
    (
        trials[
            "has_phase_2"
        ]
        |
        trials[
            "has_phase_3"
        ]
    )
)


# Emerging pipeline:
# useful later for early-stage competitive intelligence

trials[
    "is_emerging_pipeline"
] = (
    trials[
        "is_target_company_lead"
    ]
    &
    trials[
        "is_interventional"
    ]
    &
    trials[
        "has_phase_1"
    ]
)


# Operationally active pipeline view
ACTIVE_STATUSES = {
    "RECRUITING",
    "NOT_YET_RECRUITING",
    "ACTIVE_NOT_RECRUITING",
    "ENROLLING_BY_INVITATION",
}


trials[
    "is_active_core_pipeline"
] = (
    trials[
        "is_core_pipeline"
    ]
    &
    trials[
        "overall_status"
    ].isin(
        ACTIVE_STATUSES
    )
)


# ============================================================
# 8. Date parsing
# ============================================================

DATE_COLUMNS = [
    "start_date",
    "primary_completion_date",
    "completion_date",
    "status_verified_date",
]

for col in DATE_COLUMNS:

    trials[col] = pd.to_datetime(
        trials[col],
        errors="coerce"
    )


# ============================================================
# 9. Audit filtering funnel
# ============================================================

print("\n")
print("=" * 100)
print("COHORT FILTERING FUNNEL")
print("=" * 100)

funnel = pd.DataFrame({

    "cohort": [
        "All unique search matches",
        "Target-company lead sponsor",
        "Lead + interventional",
        "Core Phase 2/3 pipeline",
        "Active core pipeline",
        "Emerging Phase 1 pipeline",
    ],

    "n_trials": [
        len(trials),

        trials[
            "is_target_company_lead"
        ].sum(),

        (
            trials[
                "is_target_company_lead"
            ]
            &
            trials[
                "is_interventional"
            ]
        ).sum(),

        trials[
            "is_core_pipeline"
        ].sum(),

        trials[
            "is_active_core_pipeline"
        ].sum(),

        trials[
            "is_emerging_pipeline"
        ].sum(),
    ]
})

print(
    funnel.to_string(
        index=False
    )
)


# ============================================================
# 10. Company-level funnel
# ============================================================

print("\n")
print("=" * 100)
print("CORE PIPELINE BY COMPANY")
print("=" * 100)

company_summary = (
    trials.loc[
        trials[
            "is_target_company_lead"
        ]
    ]
    .groupby(
        "canonical_company",
        observed=True
    )
    .agg(
        total_lead_trials=(
            "nct_id",
            "size"
        ),

        interventional=(
            "is_interventional",
            "sum"
        ),

        phase1=(
            "has_phase_1",
            "sum"
        ),

        phase2=(
            "has_phase_2",
            "sum"
        ),

        phase3=(
            "has_phase_3",
            "sum"
        ),

        core_phase2_3=(
            "is_core_pipeline",
            "sum"
        ),

        active_core=(
            "is_active_core_pipeline",
            "sum"
        ),
    )
    .sort_values(
        "core_phase2_3",
        ascending=False
    )
)

print(
    company_summary
    .to_string()
)


# ============================================================
# 11. Core pipeline phase/status breakdown
# ============================================================

core = trials.loc[
    trials[
        "is_core_pipeline"
    ]
].copy()


print("\n")
print("=" * 100)
print("CORE PIPELINE — STATUS DISTRIBUTION")
print("=" * 100)

print(
    pd.crosstab(
        core[
            "canonical_company"
        ],
        core[
            "overall_status"
        ]
    )
    .to_string()
)


print("\n")
print("=" * 100)
print("CORE PIPELINE — PHASE DISTRIBUTION")
print("=" * 100)

phase_rows = []

for _, row in core.iterrows():

    for phase in row["phases"]:

        phase_rows.append({
            "canonical_company":
                row[
                    "canonical_company"
                ],

            "phase":
                phase,
        })


phase_table = (
    pd.DataFrame(
        phase_rows
    )
    .value_counts(
        [
            "canonical_company",
            "phase"
        ]
    )
    .unstack(
        fill_value=0
    )
)

print(
    phase_table
    .to_string()
)


# ============================================================
# 12. Completeness audit for fields that matter to the copilot
# ============================================================

print("\n")
print("=" * 100)
print("CORE PIPELINE — FIELD COMPLETENESS")
print("=" * 100)

scalar_fields = [
    "brief_title",
    "official_title",
    "brief_summary",
    "detailed_description",
    "enrollment",
    "start_date",
    "completion_date",
    "eligibility_criteria",
]

for col in scalar_fields:

    missing_pct = (
        core[col]
        .isna()
        .mean()
        * 100
    )

    print(
        f"{col:<30} "
        f"{missing_pct:6.2f}% missing"
    )


list_fields = [
    "conditions",
    "intervention_names",
    "primary_outcomes",
    "secondary_outcomes",
    "countries",
]

for col in list_fields:

    empty_pct = (
        core[col]
        .apply(
            lambda x:
            not isinstance(x, list)
            or len(x) == 0
        )
        .mean()
        * 100
    )

    print(
        f"{col:<30} "
        f"{empty_pct:6.2f}% empty"
    )


# ============================================================
# 13. Inspect interventions before Stage 2C
# ============================================================

print("\n")
print("=" * 100)
print("CORE PIPELINE — MOST COMMON INTERVENTIONS")
print("=" * 100)

intervention_audit = (
    core[
        [
            "canonical_company",
            "nct_id",
            "intervention_names",
            "conditions",
            "brief_title",
        ]
    ]
    .explode(
        "intervention_names"
    )
)

print(
    intervention_audit
    .groupby(
        [
            "canonical_company",
            "intervention_names"
        ],
        observed=True
    )
    .size()
    .rename("trial_count")
    .reset_index()
    .sort_values(
        [
            "canonical_company",
            "trial_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby(
        "canonical_company",
        observed=True
    )
    .head(20)
    .to_string(
        index=False
    )
)


# ============================================================
# 14. Inspect conditions before Stage 2C
# ============================================================

print("\n")
print("=" * 100)
print("CORE PIPELINE — CONDITIONS")
print("=" * 100)

condition_audit = (
    core[
        [
            "canonical_company",
            "nct_id",
            "conditions",
        ]
    ]
    .explode(
        "conditions"
    )
)

print(
    condition_audit
    .groupby(
        [
            "canonical_company",
            "conditions"
        ],
        observed=True
    )
    .size()
    .rename(
        "trial_count"
    )
    .reset_index()
    .sort_values(
        "trial_count",
        ascending=False
    )
    .head(50)
    .to_string(
        index=False
    )
)


# ============================================================
# 15. Save canonical tables
#
# Convert nested lists/dicts to JSON strings for a stable
# Parquet representation.
# ============================================================

NESTED_COLUMNS = [
    "collaborators",
    "phases",
    "conditions",
    "keywords",
    "intervention_names",
    "intervention_types",
    "intervention_descriptions",
    "primary_outcomes",
    "secondary_outcomes",
    "countries",
    "states",
    "cities",
    "matched_searches",
]


def make_parquet_safe(
    dataframe
):

    output = dataframe.copy()

    for col in NESTED_COLUMNS:

        output[col] = (
            output[col]
            .apply(
                lambda x:
                json.dumps(
                    x,
                    ensure_ascii=False
                )
            )
        )

    return output


trials_to_save = (
    make_parquet_safe(
        trials
    )
)

core_to_save = (
    make_parquet_safe(
        core
    )
)


ALL_TRIALS_PATH = (
    PROCESSED_DIR
    / "clinical_trials_canonical.parquet"
)

CORE_PATH = (
    PROCESSED_DIR
    / "obesity_core_pipeline.parquet"
)


trials_to_save.to_parquet(
    ALL_TRIALS_PATH,
    index=False
)

core_to_save.to_parquet(
    CORE_PATH,
    index=False
)


print("\n")
print("=" * 100)
print("STAGE 2B COMPLETE")
print("=" * 100)

print(
    f"Canonical search corpus: "
    f"{len(trials):,} trials"
)

print(
    f"Core Phase 2/3 pipeline: "
    f"{len(core):,} trials"
)

print(
    f"\nSaved:\n"
    f"{ALL_TRIALS_PATH}\n"
    f"{CORE_PATH}"
)

print("\nObjects ready:")
print("  trials")
print("  core")
print("  company_summary")
print("  intervention_audit")
print("  condition_audit")

STAGE 2B — CANONICAL COHORT
Raw search-result rows loaded: 480
Unique trials after deduplication: 477


COHORT FILTERING FUNNEL
                     cohort  n_trials
  All unique search matches       477
Target-company lead sponsor       388
      Lead + interventional       344
    Core Phase 2/3 pipeline       175
       Active core pipeline        79
  Emerging Phase 1 pipeline       156


CORE PIPELINE BY COMPANY
                      total_lead_trials  interventional  phase1  phase2  phase3  core_phase2_3  active_core
canonical_company                                                                                          
Novo Nordisk                        210             169      83      16      64             80           27
Eli Lilly                           115             114      37      21      50             71           39
Amgen                                29              28      15       3      10             13           12
Boehringer Ingelheim                 34

In [6]:
# ============================================================
# STAGE 2C — REVISED RELEVANCE AUDIT + INTERVENTION
# NORMALIZATION + FINAL CORPUS FREEZE
# ============================================================

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd


# ============================================================
# Reload clean Stage 2B datasets
# Makes this cell safe to rerun
# ============================================================

CANONICAL_PATH = (
    PROCESSED_DIR
    / "clinical_trials_canonical.parquet"
)

CORE_STAGE2B_PATH = (
    PROCESSED_DIR
    / "obesity_core_pipeline.parquet"
)

trials = pd.read_parquet(
    CANONICAL_PATH
)

core = pd.read_parquet(
    CORE_STAGE2B_PATH
)

print(
    f"Reloaded clean Stage 2B data: "
    f"{len(trials):,} total trials | "
    f"{len(core):,} Phase 2/3 trials"
)

print("STAGE 2C — REVISED RELEVANCE + NORMALIZATION")
print("=" * 100)


# ============================================================
# 2. Handle list columns whether objects are in-memory lists
# or JSON strings reloaded from Parquet
# ============================================================

LIST_COLUMNS = [
    "collaborators",
    "phases",
    "conditions",
    "keywords",
    "intervention_names",
    "intervention_types",
    "intervention_descriptions",
    "primary_outcomes",
    "secondary_outcomes",
    "countries",
    "states",
    "cities",
    "matched_searches",
]


def parse_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:
            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return []


for dataframe in [trials, core]:

    for col in LIST_COLUMNS:

        if col in dataframe.columns:

            dataframe[col] = (
                dataframe[col]
                .apply(parse_list)
            )


# ============================================================
# 3. Text helpers
# ============================================================

def scalar_text(x):

    if x is None:
        return ""

    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass

    return str(x)



def nested_text(x):

    if not isinstance(x, list):
        return ""

    pieces = []

    for item in x:

        if isinstance(item, dict):

            for value in item.values():

                if value is not None:
                    pieces.append(
                        str(value)
                    )

        else:

            pieces.append(
                str(item)
            )

    return " ".join(pieces)



def combined_text(*parts):

    return (
        " ".join(
            str(p)
            for p in parts
            if p
        )
        .lower()
    )


# ============================================================
# 4. Evidence patterns
#
# Important change:
#
# PRIMARY WEIGHT ENDPOINT takes precedence over presence of a
# comorbidity.
#
# Therefore:
#
# obesity + OSA + primary weight-loss endpoint
#       -> direct_obesity
#
# obesity + OSA + no meaningful weight endpoint
#       -> obesity_comorbidity
# ============================================================

OBESITY_PATTERN = re.compile(
    r"\b("
    r"obesity|obese|overweight|"
    r"weight management|weight reduction|"
    r"weight loss"
    r")\b",
    flags=re.IGNORECASE
)


WEIGHT_ENDPOINT_PATTERN = re.compile(
    r"\b("
    r"body weight|"
    r"change in weight|"
    r"change from baseline.*weight|"
    r"percent(?:age)? change.*weight|"
    r"weight reduction|"
    r"weight loss|"
    r"body mass index|"
    r"\bbmi\b|"
    r"participants.*(?:5|10|15|20)%.*weight"
    r")\b",
    flags=re.IGNORECASE
)


COMORBIDITY_PATTERN = re.compile(
    r"\b("
    r"type 1 diabetes|"
    r"type 2 diabetes|"
    r"diabetes mellitus|"
    r"sleep apnea|sleep apnoea|\bosa\b|"
    r"heart failure|\bhfpef\b|\bhfmref\b|"
    r"chronic kidney disease|\bckd\b|"
    r"osteoarthritis|"
    r"cardiovascular disease|"
    r"atherosclerotic|"
    r"hypertension|"
    r"liver fat|"
    r"steatohepatitis|\bnash\b|\bmash\b|"
    r"crohn"
    r")\b",
    flags=re.IGNORECASE
)


# ============================================================
# 5. Extract relevance evidence for every Phase 2/3 trial
# ============================================================

def relevance_signals(row):

    title = combined_text(
        scalar_text(
            row["brief_title"]
        ),
        scalar_text(
            row["official_title"]
        ),
    )

    conditions = (
        nested_text(
            row["conditions"]
        ).lower()
    )

    primary_outcomes = (
        nested_text(
            row["primary_outcomes"]
        ).lower()
    )

    summary = combined_text(
        scalar_text(
            row["brief_summary"]
        ),
        scalar_text(
            row["detailed_description"]
        ),
    )

    title_conditions = (
        f"{title} {conditions}"
    )

    obesity_explicit = bool(
        OBESITY_PATTERN.search(
            title_conditions
        )
    )

    obesity_in_title = bool(
        OBESITY_PATTERN.search(
            title
        )
    )

    primary_weight_endpoint = bool(
        WEIGHT_ENDPOINT_PATTERN.search(
            primary_outcomes
        )
    )

    comorbidity_explicit = bool(
        COMORBIDITY_PATTERN.search(
            title_conditions
        )
    )

    obesity_in_summary = bool(
        OBESITY_PATTERN.search(
            summary
        )
    )

    return pd.Series({
        "obesity_explicit":
            obesity_explicit,

        "obesity_in_title":
            obesity_in_title,

        "primary_weight_endpoint":
            primary_weight_endpoint,

        "comorbidity_explicit":
            comorbidity_explicit,

        "obesity_in_summary":
            obesity_in_summary,
    })


signal_df = core.apply(
    relevance_signals,
    axis=1
)

core = pd.concat(
    [
        core.reset_index(drop=True),
        signal_df.reset_index(drop=True),
    ],
    axis=1
)


# ============================================================
# 6. Revised conservative classifier
# ============================================================

def classify_trial(row):

    obesity = (
        row["obesity_explicit"]
    )

    weight_endpoint = (
        row[
            "primary_weight_endpoint"
        ]
    )

    comorbidity = (
        row[
            "comorbidity_explicit"
        ]
    )

    obesity_summary = (
        row[
            "obesity_in_summary"
        ]
    )


    # --------------------------------------------------------
    # Strongest evidence of an obesity-development trial:
    #
    # obesity / overweight explicitly defines population
    # AND body weight is a primary efficacy endpoint.
    #
    # This remains direct obesity even if OSA, CKD, etc.
    # are also studied.
    # --------------------------------------------------------

    if obesity and weight_endpoint:

        return pd.Series({
            "auto_relevance":
                "direct_obesity",

            "relevance_confidence":
                "high",

            "classification_reason":
                "Obesity/overweight explicit and primary weight-related endpoint",
        })


    # --------------------------------------------------------
    # Obesity population, but another disease appears to be
    # the primary therapeutic focus and weight is not a
    # primary endpoint.
    # --------------------------------------------------------

    if (
        obesity
        and comorbidity
        and not weight_endpoint
    ):

        return pd.Series({
            "auto_relevance":
                "obesity_comorbidity",

            "relevance_confidence":
                "high",

            "classification_reason":
                "Obesity population with comorbidity focus and no primary weight endpoint",
        })


    # --------------------------------------------------------
    # Obesity explicit, but trial objective is not obvious.
    # This is genuinely worth manual inspection.
    # --------------------------------------------------------

    if obesity:

        return pd.Series({
            "auto_relevance":
                "uncertain",

            "relevance_confidence":
                "medium",

            "classification_reason":
                "Obesity explicit but no clear primary weight endpoint",
        })


    # --------------------------------------------------------
    # Weight endpoint exists, but obesity is not explicit in
    # title/conditions.
    #
    # Could still be obesity development because CT.gov search
    # matched obesity elsewhere. Review manually.
    # --------------------------------------------------------

    if weight_endpoint:

        return pd.Series({
            "auto_relevance":
                "uncertain",

            "relevance_confidence":
                "medium",

            "classification_reason":
                "Primary weight endpoint but obesity not explicit in title/conditions",
        })


    # --------------------------------------------------------
    # Weak obesity reference only in narrative summary.
    # --------------------------------------------------------

    if obesity_summary:

        return pd.Series({
            "auto_relevance":
                "uncertain",

            "relevance_confidence":
                "low",

            "classification_reason":
                "Obesity appears only in narrative description",
        })


    # --------------------------------------------------------
    # No meaningful obesity-development evidence
    # --------------------------------------------------------

    return pd.Series({
        "auto_relevance":
            "not_obesity_program",

        "relevance_confidence":
            "high",

        "classification_reason":
            "No explicit obesity indication or primary weight endpoint",
    })


classification = core.apply(
    classify_trial,
    axis=1
)

core = pd.concat(
    [
        core,
        classification,
    ],
    axis=1
)


# ============================================================
# 7. Manual review queue
#
# ONLY genuinely uncertain trials need manual review now.
# ============================================================

REVIEW_LABELS = {
    "direct_obesity",
    "obesity_comorbidity",
    "not_obesity_program",
}


review_columns = [
    "nct_id",
    "canonical_company",
    "phases",
    "overall_status",
    "brief_title",
    "conditions",
    "intervention_names",
    "primary_outcomes",
    "brief_summary",
    "auto_relevance",
    "classification_reason",
]


review_candidates = (
    core.loc[
        core[
            "auto_relevance"
        ] == "uncertain",
        review_columns,
    ]
    .copy()
)


# ------------------------------------------------------------
# Preserve any labels already entered for trials that remain
# in the revised review queue.
# ------------------------------------------------------------

existing_labels = None

if RELEVANCE_REVIEW_PATH.exists():

    previous = pd.read_csv(
        RELEVANCE_REVIEW_PATH
    )

    if (
        "manual_relevance"
        in previous.columns
    ):

        keep_cols = [
            "nct_id",
            "manual_relevance",
        ]

        if (
            "manual_notes"
            in previous.columns
        ):
            keep_cols.append(
                "manual_notes"
            )

        existing_labels = (
            previous[
                keep_cols
            ]
            .copy()
        )


# Convert nested fields to readable JSON for CSV
for col in [
    "phases",
    "conditions",
    "intervention_names",
    "primary_outcomes",
]:

    review_candidates[col] = (
        review_candidates[col]
        .apply(
            lambda x:
            json.dumps(
                x,
                ensure_ascii=False
            )
        )
    )


if existing_labels is not None:

    review_candidates = (
        review_candidates
        .merge(
            existing_labels,
            on="nct_id",
            how="left"
        )
    )


if (
    "manual_relevance"
    not in review_candidates.columns
):

    review_candidates[
        "manual_relevance"
    ] = ""


if (
    "manual_notes"
    not in review_candidates.columns
):

    review_candidates[
        "manual_notes"
    ] = ""


review_candidates[
    "manual_relevance"
] = (
    review_candidates[
        "manual_relevance"
    ]
    .fillna("")
    .str.strip()
)


review_candidates[
    "manual_notes"
] = (
    review_candidates[
        "manual_notes"
    ]
    .fillna("")
)


# Validate any existing manual labels
invalid = (
    review_candidates.loc[
        (
            review_candidates[
                "manual_relevance"
            ] != ""
        )
        &
        ~review_candidates[
            "manual_relevance"
        ].isin(
            REVIEW_LABELS
        )
    ]
)


if len(invalid):

    raise ValueError(
        "Invalid manual relevance label. "
        "Allowed values are: "
        f"{sorted(REVIEW_LABELS)}"
    )


review_candidates.to_csv(
    RELEVANCE_REVIEW_PATH,
    index=False
)


# ============================================================
# 8. Apply manual labels if available
# ============================================================

manual_lookup = (
    review_candidates[
        [
            "nct_id",
            "manual_relevance",
        ]
    ]
    .copy()
)


core = core.merge(
    manual_lookup,
    on="nct_id",
    how="left",
    validate="one_to_one"
)


core[
    "manual_relevance"
] = (
    core[
        "manual_relevance"
    ]
    .fillna("")
)


core[
    "final_relevance"
] = np.where(
    core[
        "manual_relevance"
    ] != "",

    core[
        "manual_relevance"
    ],

    core[
        "auto_relevance"
    ]
)


# ============================================================
# 9. Intervention normalization
#
# Only superficial normalization is automatic.
#
# We DO NOT assume development-code aliases such as
# BI 456906 == survodutide without authoritative verification.
# ============================================================

def normalize_intervention_key(name):

    if not isinstance(
        name,
        str
    ):
        return None

    x = (
        name
        .strip()
        .lower()
    )

    x = re.sub(
        r"[®™]",
        "",
        x
    )

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


PLACEBO_PATTERN = re.compile(
    r"\bplacebo\b",
    flags=re.IGNORECASE
)


intervention_rows = []

for _, row in core.iterrows():

    for name in row[
        "intervention_names"
    ]:

        if not name:
            continue

        intervention_rows.append({
            "nct_id":
                row["nct_id"],

            "canonical_company":
                row[
                    "canonical_company"
                ],

            "raw_intervention_name":
                name,

            "normalized_key":
                normalize_intervention_key(
                    name
                ),

            "is_placebo":
                bool(
                    PLACEBO_PATTERN.search(
                        name
                    )
                ),
        })


interventions = pd.DataFrame(
    intervention_rows
)


# Most frequently occurring capitalization/spelling becomes
# the display name for equivalent normalized strings.

name_counts = (
    interventions
    .groupby(
        [
            "normalized_key",
            "raw_intervention_name",
        ],
        observed=True
    )
    .size()
    .rename("count")
    .reset_index()
)


preferred_names = (
    name_counts
    .sort_values(
        [
            "normalized_key",
            "count",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .drop_duplicates(
        "normalized_key"
    )
    .set_index(
        "normalized_key"
    )[
        "raw_intervention_name"
    ]
    .to_dict()
)


interventions[
    "canonical_intervention_name"
] = (
    interventions[
        "normalized_key"
    ]
    .map(
        preferred_names
    )
)


# ============================================================
# 10. Explicit verified aliases
#
# Keep empty until we verify aliases from authoritative
# company / registry / FDA sources.
# ============================================================

VERIFIED_ALIAS_MAP = {
    # "bi 456906": "survodutide",
}


for alias, canonical in (
    VERIFIED_ALIAS_MAP.items()
):

    mask = (
        interventions[
            "normalized_key"
        ] == alias
    )

    interventions.loc[
        mask,
        "canonical_intervention_name"
    ] = canonical


# ============================================================
# 11. Save intervention alias audit
# ============================================================

alias_audit = (
    interventions[
        [
            "raw_intervention_name",
            "normalized_key",
            "canonical_intervention_name",
            "is_placebo",
        ]
    ]
    .value_counts()
    .rename(
        "occurrences"
    )
    .reset_index()
    .sort_values(
        [
            "is_placebo",
            "occurrences",
        ],
        ascending=[
            True,
            False,
        ]
    )
)


alias_audit.to_csv(
    INTERVENTION_ALIAS_PATH,
    index=False
)


# ============================================================
# 12. Attach canonical active-drug names to trials
#
# Placebos stay in the raw intervention field but are excluded
# from the analytical active-drug list.
# ============================================================

canonical_drugs = (
    interventions.loc[
        ~interventions[
            "is_placebo"
        ]
    ]
    .groupby(
        "nct_id",
        observed=True
    )[
        "canonical_intervention_name"
    ]
    .agg(
        lambda x:
        sorted(set(x))
    )
    .rename(
        "canonical_interventions"
    )
)


core = core.merge(
    canonical_drugs,
    on="nct_id",
    how="left",
    validate="one_to_one"
)


core[
    "canonical_interventions"
] = (
    core[
        "canonical_interventions"
    ]
    .apply(
        lambda x:
        x
        if isinstance(x, list)
        else []
    )
)


# ============================================================
# 13. Revised classification results
# ============================================================

print("\n")
print("=" * 100)
print("REVISED RELEVANCE CLASSIFICATION")
print("=" * 100)

print(
    core[
        "final_relevance"
    ]
    .value_counts()
    .to_string()
)


print("\nBY COMPANY")
print("-" * 100)

print(
    pd.crosstab(
        core[
            "canonical_company"
        ],
        core[
            "final_relevance"
        ]
    )
    .to_string()
)


# ============================================================
# 14. Show why trials were classified
# ============================================================

print("\n")
print("=" * 100)
print("AUTO-CLASSIFICATION REASONS")
print("=" * 100)

print(
    core[
        "classification_reason"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 15. Manual review status
# ============================================================

pending_review = (
    review_candidates[
        "manual_relevance"
    ]
    .eq("")
    .sum()
)


print("\n")
print("=" * 100)
print("MANUAL REVIEW STATUS")
print("=" * 100)

print(
    f"Revised review queue: "
    f"{len(review_candidates):,}"
)

print(
    f"Still unlabeled: "
    f"{pending_review:,}"
)

print(
    f"\nReview file:\n"
    f"{RELEVANCE_REVIEW_PATH}"
)


# ============================================================
# 16. Build primary obesity-development corpus
# ============================================================

obesity_development = (
    core.loc[
        core[
            "final_relevance"
        ]
        == "direct_obesity"
    ]
    .copy()
)


print("\n")
print("=" * 100)
print("PRIMARY OBESITY DEVELOPMENT CORPUS")
print("=" * 100)

print(
    f"Direct obesity trials: "
    f"{len(obesity_development):,}"
)

print("\nBY COMPANY")

print(
    obesity_development[
        "canonical_company"
    ]
    .value_counts()
    .to_string()
)


print("\nBY PHASE")

phase_output = (
    obesity_development[
        [
            "canonical_company",
            "phases",
        ]
    ]
    .explode(
        "phases"
    )
)

print(
    pd.crosstab(
        phase_output[
            "canonical_company"
        ],
        phase_output[
            "phases"
        ]
    )
    .to_string()
)


# ============================================================
# 17. Parquet-safe writer
# ============================================================

SAVE_LIST_COLUMNS = [
    "collaborators",
    "phases",
    "conditions",
    "keywords",
    "intervention_names",
    "intervention_types",
    "intervention_descriptions",
    "primary_outcomes",
    "secondary_outcomes",
    "countries",
    "states",
    "cities",
    "matched_searches",
    "canonical_interventions",
]


def parquet_safe(data):

    output = data.copy()

    for col in SAVE_LIST_COLUMNS:

        if col in output.columns:

            output[col] = (
                output[col]
                .apply(
                    lambda x:
                    json.dumps(
                        x,
                        ensure_ascii=False
                    )
                )
            )

    return output


# ============================================================
# 18. Save all three analytical datasets
# ============================================================

# Complete 477-study source corpus
parquet_safe(
    trials
).to_parquet(
    FULL_CORPUS_PATH,
    index=False
)


# Annotated 175 Phase 2/3 cohort
parquet_safe(
    core
).to_parquet(
    ANNOTATED_CORE_PATH,
    index=False
)


# Current direct-obesity corpus
obesity_safe = parquet_safe(
    obesity_development
)

obesity_safe.to_parquet(
    PROVISIONAL_OBESITY_PATH,
    index=False
)


# Only freeze final corpus once ambiguous cases are reviewed

if pending_review == 0:

    obesity_safe.to_parquet(
        FINAL_OBESITY_PATH,
        index=False
    )

    DATASET_STATUS = (
        "FINAL — STAGE 2 COMPLETE"
    )

else:

    DATASET_STATUS = (
        "PROVISIONAL — MANUAL REVIEW REQUIRED"
    )


# ============================================================
# 19. Final output
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 2C OUTPUT")
print("=" * 100)

print(
    f"Dataset status: "
    f"{DATASET_STATUS}"
)

print(
    f"\nFull 477-study corpus:\n"
    f"{FULL_CORPUS_PATH}"
)

print(
    f"\nAnnotated Phase 2/3 corpus:\n"
    f"{ANNOTATED_CORE_PATH}"
)

print(
    f"\nProvisional obesity corpus:\n"
    f"{PROVISIONAL_OBESITY_PATH}"
)

if pending_review == 0:

    print(
        f"\nFINAL obesity corpus:\n"
        f"{FINAL_OBESITY_PATH}"
    )

else:

    print(
        "\nTo finish Stage 2:"
        "\n1. Open trial_relevance_review.csv"
        "\n2. Review ONLY the remaining ambiguous trials"
        "\n3. Fill manual_relevance with one of:"
        "\n   direct_obesity"
        "\n   obesity_comorbidity"
        "\n   not_obesity_program"
        "\n4. Rerun this cell"
    )


print(
    f"\nAlias audit:\n"
    f"{INTERVENTION_ALIAS_PATH}"
)

print("\nObjects retained:")
print("  core")
print("  obesity_development")
print("  interventions")
print("  alias_audit")
print("  review_candidates")

Reloaded clean Stage 2B data: 477 total trials | 175 Phase 2/3 trials
STAGE 2C — REVISED RELEVANCE + NORMALIZATION


REVISED RELEVANCE CLASSIFICATION
final_relevance
direct_obesity         139
obesity_comorbidity     34
not_obesity_program      2

BY COMPANY
----------------------------------------------------------------------------------------------------
final_relevance       direct_obesity  not_obesity_program  obesity_comorbidity
canonical_company                                                             
Amgen                              8                    1                    4
Boehringer Ingelheim               7                    0                    4
Eli Lilly                         54                    1                   16
Novo Nordisk                      70                    0                   10


AUTO-CLASSIFICATION REASONS
classification_reason
Obesity/overweight explicit and primary weight-related endpoint             133
Obesity population with comorbidit